# Road Traffic Accidents in Finland 2015–2024
## Weather, season and accident severity — a CRISP-DM analysis

**Course:** Datan käsittely ja koneoppiminen · **Process model:** CRISP-DM

| Phase | Section |
|---|---|
| 1 | Business Understanding |
| 2 | Data Understanding (accidents + FMI weather) |
| 3 | Data Preparation |
| 4 | Modeling |
| 5 | Evaluation |
| 6 | Deployment |

**Data sources**
- **Road traffic accidents:** Statistics Finland (Tilastokeskus) WFS service, layers `tieliikenne_2015 … tieliikenne_2024`. One row = one accident that led to injury or death, with year, month, hour, severity, type, involved vehicle groups and EPSG:3067 coordinates.
- **Weather:** Finnish Meteorological Institute (FMI) open data, daily observations from all stations in Finland.

**Folder layout expected by this notebook**
```
traffic_accidents_analysis.ipynb
datasets/
    tieliikenne_2015.csv ... tieliikenne_2024.csv
    fmi_cache/            <- created automatically (one small CSV per downloaded month)
```

## 1. Business Understanding

### 1.1 Background and objective
Road safety work needs to know **when** accident risk is high and **what makes an accident severe**. A common belief is that winter conditions (cold, snow, ice) are the main trigger of severe accidents. We test this belief with data instead of assuming it.

An important limitation shapes the whole project: the accident data contain **year and month but no exact date**. Every link between accidents and weather therefore has to be made at **monthly resolution**.

### 1.2 Research questions
- **RQ1 – month level:** Do monthly weather conditions (mean temperature, precipitation, snow depth) help predict the *number* of accidents in a month, once the long-term decline and the normal seasonal pattern are taken into account?
- **RQ2 – accident level:** Can the *severity* of an accident (fatal or serious injury vs. other injury) be predicted from the accident's characteristics, and does adding the monthly weather improve the prediction?

### 1.3 Hypotheses
- **H1:** Months with harsher winter weather (cold, snow) have a higher share of severe accidents.
- **H2:** Weather *anomalies* (a month colder, wetter or snowier than normal for that calendar month) explain part of the month-to-month variation in accident counts.

### 1.4 Success criteria
- **RQ1:** the model with weather has a lower mean absolute error (MAE) than the same model without weather on months that were not used for training (rolling-origin validation and the 2023–2024 hold-out).
- **RQ2:** the model with weather has a higher ROC-AUC / PR-AUC than the model without weather, and the confidence interval of the difference excludes zero. The severe class (roughly 1 accident in 7) must really be detected – a model that always answers "not severe" is not a success even if its accuracy looks high.

### 1.5 Constraints and assumptions
- Monthly resolution only; weather is a **national monthly aggregate** (not the weather at the accident site).
- Only accidents that led to injury or death are in the data, so we model *injury accidents*, not all collisions.
- No traffic-volume (exposure) data are available; the number of accidents depends on how much people drive.

## 2. Data Understanding


### 2.1 Setup


The project uses data from two sources. The road traffic accident data from **Statistics Finland** is provided as yearly CSV files covering the period from 2015 to 2024. The available files are loaded from the local datasets directory and combined into a single DataFrame.


The weather data is retrieved from the **Finnish Meteorological Institute (FMI) Open Data service** using the stored query. Daily observations on the weather are collected from the weather stations across the Finland. The analysis focuses on daily mean temperatures, precipitation and snow depth.

To make the data collection more efficient, the downloaded **FMI data** is stored locally under the **datasets/fmi_cache/**. If a file is already available, is it loaded from the cache instead of being downloaded again.

At this stage, **the two datasets** are loaded and examined separately. Their structures, variables, datatypes and missing values are inspected before proceeding to data preparation and integration.

In [ ]:
# 1. Install and verify required spatial and ML libraries
# %pip install geopandas contextily requests fmiopendata mlxtend scikit-learn

# 2. Import core engines
import glob
import os
import time
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
import contextily as ctx
import datetime as dt

from fmiopendata.wfs import download_stored_query

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, PoissonRegressor
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (mean_absolute_error, roc_auc_score, average_precision_score,
                             classification_report, ConfusionMatrixDisplay, RocCurveDisplay)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 40)

RANDOM_STATE = 69
DATA_DIR = "datasets"            # folder that holds tieliikenne_YYYY.csv
YEARS = list(range(2015, 2025))
TEST_YEARS = [2023, 2024]        # final hold-out: the two most recent years (time-based split)
print("Libraries imported.")

### 2.2 Accident data (Statistics Finland)
All ten yearly files are loaded and stacked. Columns (identical in every file):

| Column | Meaning |
|---|---|
| `vvonn`, `kkonn` | year and month of the accident (no day) |
| `kello` | one-hour interval, format `HH.00-HH.59` |
| `vakav` | severity: 1 = fatal, 2 = injury, 3 = serious injury |
| `onntyyppi` | accident type 0–9 (same/opposite/crossing directions, pedestrian, run-off-road, other) |
| `lkmhapa … lkmmuukulk` | number of involved road users per group (cars and vans, buses and trucks, pedestrians, bicycles, mopeds, motorcycles, others) |
| `x`, `y` | location, EPSG:3067 (ETRS-TM35FIN) |

In [ ]:
files = sorted(glob.glob(os.path.join(DATA_DIR, "tieliikenne_20*.csv")))
print("Files found:", [os.path.basename(f) for f in files])
assert len(files) == len(YEARS), "Expected one file per year 2015-2024"

df_raw = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

print("\nShape (rows, columns):", df_raw.shape)
print("\nData types:")
print(df_raw.dtypes)
display(df_raw.head())

df_accidents = df_raw

In total, there are 17 columns of data in the dataset. 

In [ ]:
print("Missing values")
print(df_raw.isna().sum()[lambda s: s > 0])
print("\nRows with missing 'kello' per year:")
print(df_raw[df_raw["kello"].isna()]["vvonn"].value_counts().sort_index())
print("\nDuplicate FIDs:", df_raw["FID"].duplicated().sum())

Variable (`kello`) is the only one with missing values, with a total of 16.

In [ ]:
# Class variable codebook
SEVERITY_LABELS = {1: "fatal", 2: "injury", 3: "serious injury"}
ACCIDENT_TYPES = {
    0: "same direction (straight)",
    1: "same direction (turning)",
    2: "opposite directions (straight)",
    3: "opposite directions (turning)",
    4: "crossing directions (straight)",
    5: "crossing directions (turning)",
    6: "pedestrian (crossing)",
    7: "pedestrian (elsewhere)",
    8: "run off the road",
    9: "other",
}

counts = pd.DataFrame({
    "severity": df_raw["vakav"].map(SEVERITY_LABELS).value_counts(),
})
counts["share_%"] = (counts["severity"] / len(df_raw) * 100).round(1)
display(counts)

type_counts = df_raw["onntyyppi"].map(ACCIDENT_TYPES).value_counts()
display(type_counts.to_frame("accidents"))

### 2.3 Accidents over time
Two patterns matter for modeling: a **strong long-term decline** and a **seasonal cycle**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))

per_year = df_raw.groupby("vvonn").size()
axes[0].bar(per_year.index, per_year.values, color="steelblue")
axes[0].set_title("Accidents per year")
axes[0].set_xlabel("Year"); axes[0].set_ylabel("Accidents")

heat = df_raw.groupby(["vvonn", "kkonn"]).size().unstack()
sns.heatmap(heat, annot=True, fmt="d", cmap="YlOrRd", cbar=False, ax=axes[1])
axes[1].set_title("Accidents per year and month")
axes[1].set_xlabel("Month"); axes[1].set_ylabel("Year")

plt.tight_layout(); plt.show()

print(f"Change 2015 -> 2024: {per_year.iloc[0]} -> {per_year.iloc[-1]} "
      f"({(per_year.iloc[-1] / per_year.iloc[0] - 1) * 100:.0f} %)")
print("Month with most accidents (all years):", df_raw.groupby('kkonn').size().idxmax())
print("Month with fewest accidents (all years):", df_raw.groupby('kkonn').size().idxmin())

In [ ]:
# Share of severe accidents (fatal or serious injury) by year and by month
is_severe = df_raw["vakav"].isin([1, 3])
share_year = is_severe.groupby(df_raw["vvonn"]).mean() * 100
share_month = is_severe.groupby(df_raw["kkonn"]).mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(share_year.index, share_year.values, marker="o", color="crimson")
axes[0].set_title("Severe accidents, % of all accidents, by year")
axes[0].set_xlabel("Year"); axes[0].set_ylim(0, None)
axes[1].plot(share_month.index, share_month.values, marker="o", color="crimson")
axes[1].set_title("Severe accidents, % of all accidents, by month")
axes[1].set_xlabel("Month"); axes[1].set_ylim(0, None)
plt.tight_layout(); plt.show()

print(f"Overall share of severe accidents: {is_severe.mean() * 100:.1f} %")
print("Lowest month:", share_month.idxmin(), f"({share_month.min():.1f} %) | highest month:",
      share_month.idxmax(), f"({share_month.max():.1f} %)")

### 2.4 Weather data (Finnish Meteorological Institute)
**Source:** FMI open data WFS, stored query `fmi::observations::weather::daily::multipointcoverage` (daily observations from all weather stations in the bounding box of Finland).

**Variables requested**

| FMI code | Meaning |
|---|---|
| `tday` | daily mean temperature (°C) |
| `rrday` | daily precipitation (mm) |
| `snow` | snow depth (cm) |

**How the download works and why**
- The query is sent **one calendar month at a time**. A request for a whole year over all of Finland is too large for the service and fails or times out.
- Each downloaded month is saved to `datasets/fmi_cache/`. If a download is interrupted or a month fails, just **run the cell again**: finished months are read from disk, only missing ones are fetched.
- Parameter names are matched by keyword (`temperature`, `precip`, `snow`) and the function stops with a clear message if the temperature cannot be found, so a naming change in the FMI service cannot silently produce an empty dataset.
- If a full-month request fails, the function automatically retries it as two half-month requests.

In [ ]:
FMI_QUERY = "fmi::observations::weather::daily::multipointcoverage"
FMI_BBOX = "19,59,32,71"            # lon_min, lat_min, lon_max, lat_max = all of Finland
FMI_PARAMS = "tday,rrday,snow"      # mean temperature, precipitation, snow depth
WEATHER_DIR = os.path.join(DATA_DIR, "fmi_cache")
os.makedirs(WEATHER_DIR, exist_ok=True)


def _find_value(params, *keywords):
    # Return the value of the first FMI parameter whose name contains one of the keywords.
    for name, item in params.items():
        if any(k in name.lower() for k in keywords):
            value = item.get("value")
            return np.nan if value is None else float(value)
    return np.nan


def _download_range(start, end, retries=3, pause=5):
    # Download one time range from FMI with retries. Returns obs.data or None.
    for attempt in range(1, retries + 1):
        try:
            result = download_stored_query(
                FMI_QUERY,
                args=[f"bbox={FMI_BBOX}",
                      f"starttime={start:%Y-%m-%dT%H:%M:%SZ}",
                      f"endtime={end:%Y-%m-%dT%H:%M:%SZ}",
                      f"parameters={FMI_PARAMS}"],
            )
            ## TODO: aikaisemmin if result.data: palautetaan kaikki data
            if result:
                return result.data, result.location_metadata
            reason = "empty response"
        except Exception as err:
            reason = f"{type(err).__name__}: {str(err)[:80]}"
        print(f"[{reason}; attempt {attempt}]", end=" ")
        time.sleep(pause * attempt)
    return None


def fetch_fmi_month(year, month):
    # Daily weather for one calendar month -> DataFrame(date, station, tday, rrday, snow, location).
    start = pd.Timestamp(year, month, 1)
    end = start + pd.offsets.MonthBegin(1)

    data_parts = []
    metadata_parts = []
    data, location_metadata = _download_range(start, end) # TODO: this is new
    if data is not None:
        data_parts.append(data)
        metadata_parts.append(location_metadata) # TODO: this is new

    else:                                   # fall back to two half-month requests
        middle = start + pd.Timedelta(days=15)
        for a, b in [(start, middle), (middle, end)]:
            part, metadata_part = _download_range(a, b, retries=2)
            if part is None:
                return None
            data_parts.append(part)
            metadata_parts.append(metadata_part) # TODO: this is new

    # TODO: this is new
    # flatten list of dicts
    location_dict = {}
    for location_datas in metadata_parts:
        for key, value in location_metadata.items():
            location_dict[key] = value


    rows, example_keys = [], None
    for data in data_parts:
        for timestamp, stations in data.items():
            for station, params in stations.items():
                example_keys = example_keys or list(params.keys())
                rows.append({
                    "date": pd.Timestamp(timestamp).normalize(),
                    "station": station,
                    "tday": _find_value(params, "temperature"),
                    "rrday": _find_value(params, "precip", "rain"),
                    "snow": _find_value(params, "snow"),
                    "location" : location_dict[station] # TODO: this is new
                })
    df = pd.DataFrame(rows)
    if df["tday"].isna().all():
        raise RuntimeError("Temperature not found in the FMI response. "
                           f"Parameter names returned by FMI: {example_keys}")

    return df.drop_duplicates(["date", "station"])


def load_weather_month(year, month):
    # Read a month from the local cache or download it (and cache it).
    path = os.path.join(WEATHER_DIR, f"fmi_daily_{year}-{month:02d}.csv")
    if os.path.exists(path):
        return pd.read_csv(path, parse_dates=["date"])
    print(f"FMI {year}-{month:02d}: downloading ...", end=" ")
    part = fetch_fmi_month(year, month)
    if part is None:
        print("FAILED")
        return None
    part.to_csv(path, index=False)
    print(f"{len(part)} rows, {part['station'].nunique()} stations")
    return part

In [ ]:
weather_parts, failed_months = [], []
for year in YEARS:
    for month in range(1, 13):
        part = load_weather_month(year, month)
        if part is None:
            failed_months.append(f"{year}-{month:02d}")
        else:
            weather_parts.append(part)

if failed_months:
    raise RuntimeError(f"{len(failed_months)} months could not be downloaded: {failed_months}. "
                       "Run this cell again - finished months are cached and will not be re-downloaded.")

df_weather_daily = pd.concat(weather_parts, ignore_index=True).drop_duplicates(["date", "station"])
print("Daily station observations:", df_weather_daily.shape)
display(df_weather_daily.head())

In [ ]:
# KORJATTU LAATUTARKASTUS: Rajataan data tiukasti vuosiin 2015-2024
w = df_weather_daily.copy()

# MASTER FIX: Poistetaan vuoden 2016 vuotanut ylimääräinen päivä, jotta kuukausien määrä on tasan 120
w['year_check'] = w['date'].dt.year
w = w[w['year_check'].isin(YEARS)]

# Lasketaan kattavuus uudestaan siivotusta datasta
cover = (w.assign(vuosi=w["date"].dt.year, kuukausi=w["date"].dt.month)
          .groupby(["vuosi", "kuukausi"])
          .agg(days=("date", "nunique"), stations=("station", "nunique")))

print("--- FMI Säädatan Laatutarkastus ---")
print("Months covered:", len(cover), "(odotettu: 120)")
print("Days with data per month  : min", cover["days"].min(), "| max", cover["days"].max())
print("Stations per month        : min", cover["stations"].min(), "| max", cover["stations"].max())
print("\nShare of missing values:")
print(w[["tday", "rrday", "snow"]].isna().mean().round(3))
print("\nValue ranges:")
display(w[["tday", "rrday", "snow"]].describe().T)

# Nyt nämä assertit menevät heittämällä läpi ilman virheitä!
assert len(cover) == 120, "Some months are missing from the weather data"
assert cover["days"].min() >= 28, "A month has fewer than 28 days of observations"
print("\n Kaikki tarkastukset menivät läpi! Data on puhdas ja kuukausia on tasan 120!")


**Notes on the weather values**
- FMI codes a trace amount of precipitation (and "no snow") as **−1**. These values are set to 0 in the preparation phase.
- Snow depth is measured only at some stations and mostly in winter. In summer it is empty; we treat a month without any snow observation as 0 cm.
- The number of reporting stations changes over time, which is a source of small inconsistencies in the national average.

### 2.X Weather measurement locations




In [ ]:
df_weather_daily

### 3. Data Preparation
Since raw accident records represent single events, direct continuous linear modeling on row levels is mathematically invalid. We perform an **Aggregation step** to compute total accidents grouped by month, establishing a true continuous numeric target variable suitable for Logistic Regression modeling.

In [ ]:
# TODO jostain syystä df_accidents poistettu... tämä korjaa tilapäisesti
df_accidents = df_raw

### 3.1 Accident data: readable names, hour of day, accident type, target

In [ ]:
# Readable column names for presentation
readable_col_names =  {
"vvonn" : "vuosi",
"kkonn" : "kuukausi",
"kello" : "kello",
"vakav" : "vakavuus",
"onntyyppi" : "onnettomuustyyppi",
"lkmhapa" : "henkilö- ja pakettiautot",
"lkmlaka" : "linja- ja kuorma-autot",
"lkmjk" : "jalankulkijat",
"lkmpp" : "polkupyörät ja kevyet sähköajoneuvot",
"lkmmo" : "mopot",
"lkmmp" : "moottoripyörät",
"lkmmuukulk" : "muut kulkuneuvot",
"x" : "x",
"y" : "y"
}
#
df_accidents.rename( columns=readable_col_names, inplace=True)

### Converting class variables to 1-hot encoded columns

In [ ]:
accident_types_raw = {
0 : "samat ajosuunnat (ajo suoraan)",
1 : "samat ajosuunnat (ajo kääntyen)",
2 : "vastakkaiset ajosuunnat (ajo suoraan)",
3 : "vastakkaiset ajosuunnat (ajo kääntyen)",
4 : "risteävät ajosuunnat (ajo suoraan)",
5 : "risteävät ajosuunnat (ajo kääntyen)",
6 : "jalankulkijaonnettomuus (suojatiellä)",
7 : "jalankulkijaonnettomuus (muualla)",
8 : "tieltä suistuminen",
9 : "muu onnettomuus"
}

accident_types_columns = {
0 : "type0",
1 : "type1",
2 : "type2",
3 : "type3",
4 : "type4",
5 : "type5",
6 : "type6",
7 : "type7",
8 : "type8",
9 : "type9"
}
# Create columns 'typeX' and assign boolean 1 or 0
for type_number, column_name in accident_types_raw.items():
    df_accidents[column_name] = [1 if accident_type == type_number else 0 for accident_type in df_accidents['onnettomuustyyppi']]

# Drop onnettomuustyyppi column
df_accidents.drop('onnettomuustyyppi', axis=1, inplace=True)

df_accidents.head(1)


In [ ]:
# Convert accident lethality to 0 or 1
#df_accidents['vakavuus'] = [1 if vakavuus == 1 else 0 for vakavuus in df_accidents['vakavuus']]

df_accidents['vakavuus'] = df_accidents['vakavuus'].isin([1, 3]).astype(int)
df_accidents

### Converting cyclical time variable to linear difference from midday

The hours are are in String datatype and in format XX.00-XX.59.

We take the first two characters of each string in column 'kello' and convert that to float. We convert it into float as there is a one missing value **(NaN)** that cannot be an Integer.

After dropping the missing row, conversion to whole numbers is done with **.astype(int)**

In [ ]:
df_accidents['tunti'] = df_accidents['kello'].str[:2].astype(float)

# drop the row(s) with missing time, then make it an integer
df_accidents = df_accidents.dropna(subset=['tunti'])
df_accidents['tunti'] = df_accidents['tunti'].astype(int)

# Subtracts each accident's hour from 12 and takes the absolute value
df_accidents['tunti'] = (12 - df_accidents['tunti']).abs()

# Drop the column 'kello'.
df_accidents = df_accidents.drop(columns=['kello'])

In [ ]:
# Spatial Transformation - Convert to GeoDataFrame using Finnish coordinate system (EPSG:3067)
accidents_gdf = gpd.GeoDataFrame(
    df_accidents,
    geometry=gpd.points_from_xy(df_accidents.x, df_accidents.y),
    crs="EPSG:3067"
)

print(" Data loaded and transformed successfully into Spatial GeoDataFrame!")
display(accidents_gdf.head())

### More

In [ ]:
# Drop id
df_accidents.drop('id', axis=1, inplace=True)

# Drop unnecessary string fields
df_accidents.drop('FID', axis=1, inplace=True)
df_accidents.drop('geom', axis=1, inplace=True)


# TODO: Delete this removal later, as of now 2024 holds no value
df_accidents.drop('vuosi', axis=1, inplace=True)
df_accidents.drop('kuukausi', axis=1, inplace=True)
df_accidents.drop('x', axis=1, inplace=True)
df_accidents.drop('y', axis=1, inplace=True)


### 3.2 Weather: from daily station values to one value per month
1. Clean the codes (−1 = trace → 0).
2. Average over all stations to get one national value **per day**.
3. Aggregate the days into **monthly features**: mean temperature, total precipitation, mean snow depth and the share of days with a mean temperature below 0 °C.

In [ ]:
# 1. Create a copy of the raw daily weather table
d_stations = df_weather_daily.copy()

# Clean up trace amounts and missing codes as we did before
d_stations["rrday"] = d_stations["rrday"].clip(lower=0)
d_stations["snow"] = d_stations["snow"].clip(lower=0)

# Extract year and month fields from the datetime index
d_stations["vuosi"] = d_stations["date"].dt.year
d_stations["kuukausi"] = d_stations["date"].dt.month

# 2. Aggregation Phase: Group by Year, Month, Station Name, and Location coordinates!
# This collapses the 10-minute/daily records into a clean Monthly Station Dataset.
df_weather_monthly_stations = d_stations.groupby(
    ["vuosi", "kuukausi", "station", "location"], 
    as_index=False
).agg(
    keski_lampotila=("tday", "mean"),       # Monthly mean temp for THIS specific station
    sade_summa=("rrday", "sum"),            # Monthly total rain for THIS specific station
    lumensyvyys=("snow", "mean"),          # Monthly mean snow depth for THIS specific station
    pakkaspaivat_osuus=("tday", lambda s: (s.dropna() < 0).mean()) # Frost days share for THIS station
)

# Handle missing snow values for the stations safely
df_weather_monthly_stations["lumensyvyys"] = df_weather_monthly_stations["lumensyvyys"].fillna(0)

print("Monthly weather dataset with unique locations constructed successfully!")
print(f"Dataset Shape (Monthly Station Records): {df_weather_monthly_stations.shape}")

# Display the first 14 rows to let Abu Shahab see the station names and coordinates
display(df_weather_monthly_stations.head(14))


Convert daily FMI observations into monthly averages
WHILE KEEPING the unique Station Name and its Geographic Location!


In [ ]:
WX_FEATURES = ["keski_lampotila", "sade_summa", "lumensyvyys", "pakkaspaivat_osuus"]

fig, axes = plt.subplots(2, 2, figsize=(14, 7), sharex=True)
x = pd.to_datetime(dict(year=df_weather_monthly["vuosi"], month=df_weather_monthly["kuukausi"], day=1))
titles = ["Mean temperature (°C)", "Total precipitation (mm, station average)",
          "Mean snow depth (cm)", "Share of days below 0 °C"]
for ax, col, title in zip(axes.ravel(), WX_FEATURES, titles):
    ax.plot(x, df_weather_monthly[col], color="teal")
    ax.set_title(title)
plt.tight_layout(); plt.show()

### Functions for converting location strings to (lon,lat) tuples


In [ ]:
# TODO: tähän tulee funktiot jotka muuntaa stringit geopy yhteensopiviksi koordinaateiksi
from geopy import distance
from pyproj import Transformer

def epsg_string_to_lonlat(string):
        string = string.strip("{}")
        lon_lat_dict = dict( item.replace("'","").split(": ") for item in string.split(", ") )
        print( lon_lat_dict)
        x = lon_lat_dict["longitude"]
        y = lon_lat_dict["latitude"]
        transformer = Transformer.from_crs(f"EPSG:3067", "EPSG:4326", always_xy=True)
        lon, lat = transformer.transform(x, y)
        return lon, lat


def point_string_to_lonlat(string):
    # String like : POINT (397544.8774 6675865.1006)
    lon, lat = string.strip("POINT ()").split(" ")
    lon = float(lon)
    lat = float(lat)
    return lon, lat

teststeing = "POINT (397544.8774 6675865.1006)"
teststeing2 = "{'fmisid': 100907, 'latitude': 60.12467, 'longitude': 19.90361}"

print( point_string_to_lonlat(teststeing) )
print( epsg_string_to_lonlat(teststeing2) )







### 3.X Separating training and testing data

The data is divided into training (75%) and testing (25%). To guarantee same results across every run, a fixed **random_state** is applied.

In [ ]:
from sklearn.model_selection import train_test_split

y = df_accidents['vakavuus']
X = df_accidents.drop(columns= 'vakavuus')

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=69
)

In [ ]:
# TODO: You can remove this later
X_train

In [ ]:
# TODO: You can remove this later
X_test

In [ ]:
y_train

In [ ]:
y_test

## 3.X Feature scaling

To prepare the features for moding, we standardize them using **Scikit-Learn's StandardScaler**. This transforms the features to have a mean of 0 and a standard deviation of 1.

The scaling is done due to Logistic Regression is sensitive to the scale of input features. Features with larger numeric ranges could otherwise dominate the model's coefficients.

Scaler is fitted strictly on the training set **X_train** to learn its mean and variance. These values are then applied to the test set **X_test**, ensuring no information from the testing data leaks into the training process.

In [ ]:
## Scale features

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit the scaler on X_train data only to learn its mean and variance
X_train_scaled = scaler.fit_transform(X_train)
# Scale X_test using the exact same mean and variance learned from X_train
X_test_scaled = scaler.transform(X_test)

# Wrap results back into a DataFrame
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_train.columns)

# assert the list sizes, small check.
assert len(X_train_scaled) == len(y_train) and len(X_test_scaled) == len(y_test)

print("Data Preparation Completed:")
print(f"Training set size: {X_train.shape[0]} samples.")
print(f"Testing set size: {X_test.shape[0]} samples.")

print("\n1. Without standardize features train data :")
display(X_train.head(5))

print("\n2. With standardize features train data (for the Logistic regression) :")
display(X_train_scaled.head(5))



# TODO: ADD REGRESSION MODELING AND VALIDATION BELLOW

### 4. Modeling

With the data prepared, we initialize and train a **LogisticRegression** model from Scikit-Learn. The model is train using the **X_train_scaled** and **y_train**.


In [ ]:
from sklearn.linear_model import LogisticRegression

# Model initialization and training
model = LogisticRegression(class_weight='balanced', random_state=69)
model.fit(X_train_scaled, y_train)

In [ ]:
# TODO: Move to 5. Evaluation later

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

# Train accuracy
print("Train accuracy:", model.score(X_train_scaled, y_train))
# Test accuracy
print("Test accuracy: ", model.score(X_test_scaled, y_test))

In [ ]:
pred = model.predict(X_test_scaled)
print(classification_report(y_test, pred))

ConfusionMatrixDisplay.from_predictions(
    y_test, pred,
    display_labels=['Less severe', 'Severe'],
    cmap='Blues'
)
plt.title('Logistic regression: confusion matrix')
plt.show()

In [ ]:
print(y_train.value_counts())
print(y_test.value_counts())
print(df_accidents['vakavuus'].value_counts())

### 4. X Alternative Cross Validation
Cross validation instead of split validation

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import accuracy_score

# cross-validation

y_pred = cross_val_predict(estimator=model, X=X_train_scaled, y=y_train, cv=10)

cm = confusion_matrix(y_train, y_pred)
accuracy = accuracy_score(y_train, y_pred)

print("Accuracy: %0.2f" % accuracy)
print("Confusion Matrix:\n", cm)

# visualize confusion matrix
plt.matshow(cm)
plt.title('Confusion matrix')
plt.colorbar()
plt.ylabel('True label')
plt.xlabel('Predicted label')
# include counts
for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha='center', va='center', color='red', backgroundcolor='white')
plt.show()

### 4.x FMI Weather Interface Testing
We test the FMI Open Data live streaming engine to collect auxiliary meteorological variables corresponding to the accident location spaces.

In [ ]:
# TODO: this is for testing weather data only

import pandas as pd
import datetime as dt
from fmiopendata.wfs import download_stored_query

try:
    print("Connecting to FMI Government server... Fetching raw spatial coordinates...")
    
    # 1. Fetch data from FMI using a defined bounding box (Standard multipoint query)
    obs = download_stored_query(
        "fmi::observations::weather::multipointcoverage", 
        args=["bbox=24.5,60.1,25.1,60.3"]
    )
    
    # 2. Manual dictionary parsing loop to completely bypass 'to_dataframe' error
    weather_records = []
    
    # Loop through each unique timestamp key in the raw data
    for timestamp in obs.data.keys():
        # Loop through each weather station name key at this timestamp
        for station_name in obs.data[timestamp].keys():
            station_data = obs.data[timestamp][station_name]
            
            # Extract main target parameters provided by FMI
            temp = station_data.get('Air temperature', {}).get('value', None)
            wind = station_data.get('Wind speed', {}).get('value', None)
            rh = station_data.get('Relative humidity', {}).get('value', None)
            
            # Append rows manually to our collection list
            weather_records.append({
                "time": timestamp,
                "location": station_name,
                "air_temperature": temp,
                "wind_speed": wind,
                "relative_humidity": rh
            })
            
    # 3. Construct the clean Pandas DataFrame manually!
    weather_df = pd.DataFrame(weather_records)
    
    print("Real FMI Weather database constructed successfully without errors!")
    print(f"Weather Data Shape: {weather_df.shape}")
    
    # Display the first 10 rows for data understanding validation
    display(weather_df.head(10))

except Exception as e:
    print(f"Final Engine Error Notice: {e}. Check server status.")


### 5. Evaluation

### 5.X Feature Importance

In [ ]:
from sklearn.inspection import permutation_importance

# Calculate permutation importance using the standardized model and scaled test data
result = permutation_importance(
    model,
    X_train_scaled,
    y_train,
    n_repeats=10, # TODO: what is good number of repeats?
    scoring="accuracy",
    random_state=69,
    n_jobs=-1
)

# Sort features by their mean importance score
sorted_idx = result.importances_mean.argsort()

# Plot the results
fig, ax = plt.subplots()
ax.boxplot(
    result.importances[sorted_idx].T,
    orientation="horizontal",
    tick_labels=X_train_scaled.columns[sorted_idx] #aikaisemmin df_accidents
)
ax.set_title("Permutation Feature Importance")
fig.tight_layout()
plt.show()


### 6. Deployment